# UdaPlay Part 1

Build a persistent vector database from the fifteen supplied game records. Run this notebook before Part 2, from `project/starter`. The database directory is `chromadb` and the collection is `udaplay`.

## Environment

Configure `OPENAI_API_KEY`, `OPENAI_BASE_URL`, and `TAVILY_API_KEY` using an ignored `.env` or exported environment variables. Credentials are never displayed.

In [ ]:
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [ ]:
from pathlib import Path
import json

from lib.udaplay import configure_environment, load_games, GameVectorStore

settings = configure_environment()
print("Required environment variables are configured.")
print("Embedding model:", settings["embedding_model"])

## Load and validate the dataset

Each JSON file becomes a document with the game name, platform, genre, publisher, description and release year. File stems provide stable IDs. Metadata retains the original values and a source path for citations.

In [ ]:
records = load_games(Path("games"))
assert len(records) == 15
print("Validated game records:", len(records))
print(json.dumps(records[0], indent=2, ensure_ascii=False))

## Persist the collection

The store configures OpenAI embeddings using the same key, endpoint and model in both notebooks. Upsert makes repeated ingestion safe.

In [ ]:
store = GameVectorStore(path="chromadb", create=True)
count = store.ingest(records)
assert count == 15
print(f"Collection {store.collection.name!r}: {count} documents")

## Reopen and query

Reopen the database to demonstrate persistence. Semantic search returns candidate documents; similarity alone does not establish answer quality. Part 2 evaluates that separately.

In [ ]:
reopened = GameVectorStore(path="chromadb")
assert reopened.collection.count() == 15
query = "Who published Grand Theft Auto: San Andreas?"
results = reopened.search(query)
assert results
print("Query:", query)
for result in results:
    print(json.dumps(result.model_dump(), indent=2, ensure_ascii=False))

## Part 1 result

The persistent database is ready for Part 2 after all cells above succeed. Keep `chromadb/` locally between runs. It is reproducible from the dataset and is excluded from Git.